In [234]:
using MarineHydro
using PyCall
using DimensionalData
cpt = pyimport("capytaine")

# Description of problem
h = Inf # sea depth [m]
omegas = [1.03, 1.7] # frequencies [rad/s]
beta = pi/3 # incident wave angle [rad]
t_DOFs = ["Heave"] # translational DOFs
r_DOFs = ["Pitch"] # rotational DOFs
DOFs = [t_DOFs; r_DOFs] # all DOFs

# Create Mesh object
radius = 1.5  
center = (0.0,0.0,0.0) 
len = 2.5
faces_max_radius = 0.5
cptmesh = cpt.meshes.predefined.mesh_horizontal_cylinder(
            radius=radius,
            center=center, 
            length=len, 
            faces_max_radius = faces_max_radius
            ).keep_immersed_part(inplace=true)


# Create FloatingBody object
cptbody = cpt.FloatingBody(mesh=cptmesh)
cptbody.center_of_mass = (0.0, 0.0, 0.0)
cptbody.rotation_center = (1.0, 1.0, 0.0) # off set for nonzero off-diagoinal elements
foreach(dof -> cptbody.add_translation_dof(name=dof), t_DOFs)
foreach(dof -> cptbody.add_rotation_dof(name=dof), r_DOFs)
cptbody.active_dofs = DOFs
cptbody.name = "Horizontal Cylinder"


# Setup and solve BEM problems
solver = cpt.BEMSolver()
dof_list = cptbody.active_dofs
xr = pyimport("xarray")
test_matrix = xr.Dataset(coords=Dict("omega" => omegas, "wave_direction" => [beta], "radiating_dof" => DOFs))
results = cpt.BEMSolver().fill_dataset(test_matrix, cptbody, method="direct")
 

# Get Capytaine values
A_cpt = results.added_mass
B_cpt = results.radiation_damping
F_FK_cpt = results.Froude_Krylov_force 
F_D_cpt = results.diffraction_force

Solving BEM problems ---------------------------------------- 100% 0:00:00


PyObject <xarray.DataArray 'diffraction_force' (omega: 2, wave_direction: 1,
                                       influenced_dof: 2)> Size: 64B
array([[[ -6927.50065297-1895.70173733j,  -6938.80492983 -197.25459521j]],

       [[-13045.71747135-6603.53341329j, -13531.42109187-1753.0394137j ]]])
Coordinates:
    g               float64 8B 9.81
    rho             float64 8B 1e+03
    body_name       <U19 76B 'Horizontal Cylinder'
    water_depth     float64 8B inf
    forward_speed   float64 8B 0.0
  * wave_direction  (wave_direction) float64 8B 1.047
  * omega           (omega) float64 16B 1.03 1.7
  * influenced_dof  (influenced_dof) object 16B 'Heave' 'Pitch'
    period          (omega) float64 16B 6.1 3.696
    wavenumber      (omega) float64 16B 0.1081 0.2946
    wavelength      (omega) float64 16B 58.1 21.33
Attributes:
    long_name:  Diffraction force

In [235]:
F_D_cpt

PyObject <xarray.DataArray 'diffraction_force' (omega: 2, wave_direction: 1,
                                       influenced_dof: 2)> Size: 64B
array([[[ -6927.50065297-1895.70173733j,  -6938.80492983 -197.25459521j]],

       [[-13045.71747135-6603.53341329j, -13531.42109187-1753.0394137j ]]])
Coordinates:
    g               float64 8B 9.81
    rho             float64 8B 1e+03
    body_name       <U19 76B 'Horizontal Cylinder'
    water_depth     float64 8B inf
    forward_speed   float64 8B 0.0
  * wave_direction  (wave_direction) float64 8B 1.047
  * omega           (omega) float64 16B 1.03 1.7
  * influenced_dof  (influenced_dof) object 16B 'Heave' 'Pitch'
    period          (omega) float64 16B 6.1 3.696
    wavenumber      (omega) float64 16B 0.1081 0.2946
    wavelength      (omega) float64 16B 58.1 21.33
Attributes:
    long_name:  Diffraction force

In [236]:
# Restructure MDOF marinehydro functions
mhmesh = Mesh(cptmesh)
num_panels = mhmesh.nfaces

# manually make heave dof matrix
dof_mat_1 = zeros(num_panels,3)
dof_mat_1[:,3].=1

# manually make surge dof matrix
dof_mat_2 = zeros(num_panels,3)
dof_mat_2[:,2].=1

# make dof named tuple
dofs = (dof1 = dof_mat_1, dof2 = dof_mat_2)


(dof1 = [0.0 0.0 1.0; 0.0 0.0 1.0; … ; 0.0 0.0 1.0; 0.0 0.0 1.0], dof2 = [0.0 1.0 0.0; 0.0 1.0 0.0; … ; 0.0 1.0 0.0; 0.0 1.0 0.0])

In [237]:
Dofs = NamedTuple
Dofs

NamedTuple

In [242]:
# redo floatingbody object
using LinearAlgebra: cross, dot, norm

struct FloatingBody
    mesh::Mesh
    dofs::NamedTuple 

    function FloatingBody(mesh::Mesh, dofs::NamedTuple)
        #add assert statements

        return new(mesh, dofs)
    end
end



# function FloatingBody(mesh::Mesh, rigid_dof_list::Vector{String}, rotation_center::AbstractVector)
#     dofs = Dict()

#     for dof_name in rigid_dof_list
#         if dof_name in ["Surge", "Sway", "Heave"]
#             dofs[dof_name] = translational_dofs(mesh, dof_name)
#         elseif dof_name in ["Roll", "Pitch", "Yaw"]
#             dofs[dof_name] = rotational_dofs(mesh, dof_name, rotation_center)
#         end
#     end
#     return FloatingBody(mesh, dofs)
# end

function FloatingBody(mesh::Mesh, rigid_dof_list::Vector{String}, rotation_center::AbstractVector)
    
    # generator
    dof_pairs = (Symbol(name) => if name in ["Surge", "Sway", "Heave"]
            translational_dofs(mesh, name)
        else
            rotational_dofs(mesh, name, rotation_center)
        end for name in rigid_dof_list)
            
    # convert Pair to NamedTuple using ; and splat
    dofs = (; dof_pairs...)

    return FloatingBody(mesh, dofs)
end

function translational_dofs(mesh::Mesh, dof_name::String)
    num_panels = mesh.nfaces
    dof = zeros(num_panels, 3)
    if dof_name=="Surge"
        dof[:,1] .= 1
    elseif dof_name=="Sway"
        dof[:,2] .= 1
    elseif dof_name=="Heave"
        dof[:,3] .= 1
    end
    return dof
end

function rotational_dofs(mesh::Mesh, dof_name::String, rotation_center::AbstractVector)
    face_centers = mesh.centers
    if dof_name=="Roll"
        axis_of_rot = [1, 0, 0]
    elseif dof_name=="Pitch"
        axis_of_rot = [0, 1, 0]
    elseif dof_name=="Yaw"
        axis_of_rot = [0, 0, 1]
    end
    pos_vec = face_centers .- rotation_center'
    dof_vecs = cross.(Ref(axis_of_rot), eachrow(pos_vec))
    dof = stack(dof_vecs)' # make vector of vectors into a matrix
    return dof
end

function FloatingBody(mesh::Mesh, rigid_dof_list::Vector{String})
    rotation_center = [0.0,0.0,0.0]
    for dof in rigid_dof_list
        if dof in ["Roll","Pitch","Yaw"]
            display("Setting origin as rotation center.")
        end
    end
    return FloatingBody(mesh::Mesh, rigid_dof_list::Vector{String}, rotation_center::AbstractVector)
end

FloatingBody

In [245]:
mesh = mhmesh
rotation_center = [1.0,1.0,0.0]


fb = FloatingBody(mhmesh,DOFs,rotation_center)

FloatingBody(Mesh([-1.25 -1.4623918682727355 -0.33378140093447134; -1.25 -1.4623918682727357 0.0; … ; 1.25 1.4623918682727355 -0.3337814009344717; 1.25 1.4623918682727355 -2.7755575615628914e-17], [21 33 32 12; 33 42 41 32; … ; 82 79 78 81; 59 56 55 58], [-0.9375 0.3254128043381684 -1.4257266509268143; -0.3125 0.3254128043381684 -1.4257266509268143; … ; 1.25 1.2349086887636433 -0.14092992483899916; 1.25 -1.2349086887636436 -0.14092992483899902], [0.0 0.2225209339563142 -0.9749279121818238; 0.0 0.2225209339563142 -0.9749279121818238; … ; 1.0 0.0 0.0; 1.0 0.0 0.0], [0.41722675116808966, 0.41722675116808966, 0.41722675116808966, 0.41722675116808966, 0.4172267511680895, 0.4172267511680895, 0.4172267511680895, 0.4172267511680895, 0.41722675116808955, 0.41722675116808955  …  0.054235467389694765, 0.02711773369484739, 0.02711773369484736, 0.05423546738969476, 0.05423546738969476, 0.05423546738969479, 0.08135320108454216, 0.08135320108454208, 0.13558866847423695, 0.13558866847423684], [0.45723

In [246]:
keys(fb.dofs)

(:Heave, :Pitch)

In [ ]:
# constants
g = 9.81 # acceleration due to gravity [m/s^2]
rho = 1020 # fluid density [kg/m^3]

In [162]:
# redo radiation_bc function

function radiation_bc(floatingbody::FloatingBody, omega)
    """
    Calculates the radiation boundary conditions for floating bodies at each panel.

    # Arguments
    - `floatingbody::FloatingBody`: The floating body
    - `omega`: The frequency of the incident ocean wave ~~~.

    # Returns
    - The (Neumann) radiation boundary condition values for each panel.
"""
    normals_mat = floatingbody.mesh.normals

    # generator
    bc_pairs = (dof_symbol => begin
        dof_mat = floatingbody.dofs[dof_symbol]
        -1im .* omega .* sum(normals_mat .* dof_mat, dims=2) # output
    end for dof_symbol in keys(floatingbody.dofs))    
   
    # convert Pair to NamedTuple using ; and splat
    bc = (; bc_pairs...)
    return bc
end




radiation_bc (generic function with 1 method)

In [163]:
radiation_bc(fb, 1.5)

(Heave = ComplexF64[-0.0 + 1.4623918682727357im; -0.0 + 1.4623918682727357im; … ; 0.0 - 0.0im; 0.0 - 0.0im;;], Pitch = ComplexF64[-0.0 + 2.8333842447784257im; -0.0 + 1.9193893271079656im; … ; -0.0 + 0.21139488725849875im; -0.0 + 0.21139488725849853im;;])

In [164]:
# redo integrate ressure
function integrate_pressure(floatingbody::FloatingBody, pressure)
    mesh = floatingbody.mesh

    # generator
    force_pairs = (dof_symbol => begin
        dof_mat = floatingbody.dofs[dof_symbol]
        normal_dof_amp_on_face = -sum(dof_mat .* mesh.normals, dims=2)
        sum(pressure .* normal_dof_amp_on_face .* mesh.areas) # output
    end for dof_symbol in keys(floatingbody.dofs))

    # convert Pair to NamedTuple using ; and splat
    forces = (; force_pairs...)
    return forces
end

integrate_pressure (generic function with 1 method)

In [165]:
dummy_pressure = ones(num_panels,1)
forces_tuple = integrate_pressure(fb, dummy_pressure)
vec1 = collect(ComplexF64, values(forces_tuple))
vec2 = vec1
vec_tuple = (vec1 = vec1, vec2 = vec2)
mat = hcat(values(vec_tuple)...)
real(mat)
values(forces_tuple)

(7.3119593413636785, 7.311959341363677)

In [166]:
# redo radiation calcs
function calculate_radiation_forces(floatingbody::FloatingBody, omega)
    rho = 1023
    g = 9.81

    k = omega^2 / g
    mesh = floatingbody.mesh
    S, D = assemble_matrix_wu(mesh, k)
    rad_bcs = radiation_bc(floatingbody, omega) 

    # generator
    force_vector_pairs = (dof_symbol => begin
        rad_bc = rad_bcs[dof_symbol]
        potential = solve(D, S, rad_bc)
        pressure = 1im * rho * omega * potential
        force_tuples = integrate_pressure(floatingbody, pressure)
        collect(ComplexF64, values(force_tuples))
    end for dof_symbol in keys(floatingbody.dofs))

    # convert Pair to NamedTuple using ; and splat
    force_vectors_tuple = (; force_vector_pairs...)

    # convert NamedTuple of vectors into a matrix
    rad_force_matrix = hcat(values(force_vectors_tuple)...)

    # assemble hydrodynamic coefficients 
    Added_mass_mat = real(rad_force_matrix)/omega^2
    Radiation_damping_mat = imag(rad_force_matrix)/omega

    return Added_mass_mat, Radiation_damping_mat
end

calculate_radiation_forces (generic function with 1 method)

In [189]:
A_mh = calculate_radiation_forces(fb, omegas[1])[1]

2×2 Matrix{Float64}:
 7297.91   7297.91
 7297.91  10411.7

In [168]:
A_cpt

PyObject <xarray.DataArray 'added_mass' (omega: 2, radiating_dof: 2, influenced_dof: 2)> Size: 64B
array([[[6818.23611669, 6818.23611669],
        [6818.23611669, 9951.66185184]],

       [[5305.90764351, 5305.90764351],
        [5305.90764351, 8670.68965221]]])
Coordinates:
    g               float64 8B 9.81
    rho             float64 8B 1e+03
    body_name       <U19 76B 'Horizontal Cylinder'
    water_depth     float64 8B inf
    forward_speed   float64 8B 0.0
  * omega           (omega) float64 16B 1.03 1.7
  * radiating_dof   (radiating_dof) object 16B 'Heave' 'Pitch'
  * influenced_dof  (influenced_dof) object 16B 'Heave' 'Pitch'
    period          (omega) float64 16B 6.1 3.696
    wavenumber      (omega) float64 16B 0.1081 0.2946
    wavelength      (omega) float64 16B 58.1 21.33
Attributes:
    long_name:  Added mass

In [169]:
# redo function

function FroudeKrylovForce(floatingbody::FloatingBody, ω)
    """Compute the Froude-Krylov force."""
    F_FK = Dict{Tuple{Any, String}, Any}()

    mesh = floatingbody.mesh
    pressure =  airy_waves_pressure(mesh.centers,  ω)
    forces_tuple = integrate_pressure(floatingbody::FloatingBody, pressure) 
    F_FK = collect(ComplexF64, values(forces_tuple)) # convert NamedTuple to Vector
    return F_FK  
end

FroudeKrylovForce (generic function with 1 method)

In [170]:
FroudeKrylovForce(fb, omegas[1])

2-element Vector{ComplexF64}:
  63068.81437521969 + 1.4709558352063256e-13im
 63068.814375219685 + 1832.6589348041916im

In [171]:
F_FK_cpt

PyObject <xarray.DataArray 'Froude_Krylov_force' (omega: 2, wave_direction: 1,
                                         influenced_dof: 2)> Size: 64B
array([[[63068.81437522+9.02521647e-14j, 63068.81437522+1.83265893e+03j]],

       [[49963.68175156+2.36290425e-13j, 49963.68175156+4.58554457e+03j]]])
Coordinates:
    g               float64 8B 9.81
    rho             float64 8B 1e+03
    body_name       <U19 76B 'Horizontal Cylinder'
    water_depth     float64 8B inf
    forward_speed   float64 8B 0.0
  * wave_direction  (wave_direction) float64 8B 0.0
  * omega           (omega) float64 16B 1.03 1.7
  * influenced_dof  (influenced_dof) object 16B 'Heave' 'Pitch'
    period          (omega) float64 16B 6.1 3.696
    wavenumber      (omega) float64 16B 0.1081 0.2946
    wavelength      (omega) float64 16B 58.1 21.33
Attributes:
    long_name:  Froude Krylov force

In [172]:
# redo dif force fun

function diffraction_force(floatingbody::FloatingBody,potential,omega)
    rho = 1000
    pressure = 1im*rho* potential * omega 
    forces = integrate_pressure(floatingbody::FloatingBody, pressure) 
    return forces  
end





function DiffractionForce(floatingbody::FloatingBody,ω)
    

    mesh = floatingbody.mesh
    green_functions = (
        Rankine(),
        RankineReflected(),
        GFWu(),
    )
    k = ω^2 / 9.8
    S, D = assemble_matrices(green_functions, mesh, k)
    bc = AiryBC(mesh, ω)
    potential = solve(D, S, bc)
    forces_tuple = diffraction_force(floatingbody, potential,ω)
    F_D = collect(ComplexF64, values(forces_tuple)) # convert NamedTuple to Vector
    return F_D
end

DiffractionForce (generic function with 1 method)

In [214]:
F_D_mh = DiffractionForce(fb,omegas[1])

2-element Vector{ComplexF64}:
  -284.5977449165626 - 4.288839921571355im
 -284.59779687462145 + 120.54090668974253im

In [174]:
F_D_cpt

PyObject <xarray.DataArray 'diffraction_force' (omega: 2, wave_direction: 1,
                                       influenced_dof: 2)> Size: 64B
array([[[ -6743.84717853-1903.32445886j,  -6766.41387212+1473.77719903j]],

       [[-11852.53171731-6795.65856077j, -12808.28557252+2534.67162652j]]])
Coordinates:
    g               float64 8B 9.81
    rho             float64 8B 1e+03
    body_name       <U19 76B 'Horizontal Cylinder'
    water_depth     float64 8B inf
    forward_speed   float64 8B 0.0
  * wave_direction  (wave_direction) float64 8B 0.0
  * omega           (omega) float64 16B 1.03 1.7
  * influenced_dof  (influenced_dof) object 16B 'Heave' 'Pitch'
    period          (omega) float64 16B 6.1 3.696
    wavenumber      (omega) float64 16B 0.1081 0.2946
    wavelength      (omega) float64 16B 58.1 21.33
Attributes:
    long_name:  Diffraction force

In [184]:
mat = rand(3, 3, 4)

3×3×4 Array{Float64, 3}:
[:, :, 1] =
 0.947796  0.131209  0.439206
 0.578177  0.994215  0.20845
 0.323573  0.401513  0.297852

[:, :, 2] =
 0.454633  0.834472  0.90064
 0.504005  0.232791  0.742741
 0.945664  0.487296  0.314705

[:, :, 3] =
 0.722332  0.35935   0.420691
 0.323553  0.205062  0.193712
 0.622415  0.392183  0.0392848

[:, :, 4] =
 0.643452   0.0689282  0.0219551
 0.223751   0.33825    0.141219
 0.0171785  0.204692   0.388423

In [209]:
dof_list = collect(keys(fb.dofs))
data_array = reshape(A_mh, size(A_mh)..., 1)
data_labeled = DimArray(data_array, (
        Dim{:influenced_DOFs}(dof_list), 
        Dim{:radiating_DOFs}(dof_list),
        Dim{:wave_frequency}([1.2])
    ))

┌ 2×2×1 DimArray{Float64, 3} ┐
├────────────────────────────┴─────────────────────────────────────── dims ┐
  ↓ influenced_DOFs Categorical{Symbol} [:Heave, :Pitch] ForwardOrdered,
  → radiating_DOFs Categorical{Symbol} [:Heave, :Pitch] ForwardOrdered,
  ↗ wave_frequency Sampled{Float64} [1.2] ForwardOrdered Irregular Points
└──────────────────────────────────────────────────────────────────────────┘
[:, :, 1]
 ↓ →          :Heave       :Pitch
  :Heave  7297.91      7297.91
  :Pitch  7297.91     10411.7

In [222]:
# convert added mass and damping matrices
function label_output(floatingbody::FloatingBody, data_array::Array{Float64, 3}, omegas)
    dof_list = collect(keys(floatingbody.dofs))
    num_dofs = length(dof_list)
    num_omegas = length(omegas)

    data_labeled = DimArray(data_array, (
        Dim{:influenced_DOFs}(dof_list), 
        Dim{:radiating_DOFs}(dof_list),
        Dim{:wave_frequency}(omegas)
    ))
    return data_labeled
end

# convert single frequency added mass and damping
function label_output(floatingbody::FloatingBody, data_mat::Matrix{Float64}, omega)
    data_array = reshape(data_mat, size(data_mat)..., 1)
    return label_output(floatingbody, data_array, [omega])
end

# convert excitation, FK, or diffraction force
function label_output(floatingbody::FloatingBody, data_array::Array{ComplexF64, 3}, omegas, betas)
    dof_list = collect(keys(floatingbody.dofs))
    num_dofs = length(dof_list)
    num_betas = length(betas)
    num_omegas = length(omegas)

    data_labeled = DimArray(data_array, (
        Dim{:influenced_DOFs}(dof_list), 
        Dim{:wave_frequency}(omegas),
        Dim{:wave_direction}(betas)
    ))
    return data_labeled
end

# convert single frequency and beta excitation, FK, or diffraction force
function label_output(floatingbody::FloatingBody, data_mat::Vector{ComplexF64}, omega, beta)
    data_array = reshape(data_mat, size(data_mat)..., 1, 1)
    return label_output(floatingbody::FloatingBody, data_array::Array{ComplexF64, 3}, [omega], [beta])
end



label_output (generic function with 7 methods)

In [223]:
A_mh_labeled = label_output(fb, A_mh, omegas[1])

┌ 2×2×1 DimArray{Float64, 3} ┐
├────────────────────────────┴─────────────────────────────────────── dims ┐
  ↓ influenced_DOFs Categorical{Symbol} [:Heave, :Pitch] ForwardOrdered,
  → radiating_DOFs Categorical{Symbol} [:Heave, :Pitch] ForwardOrdered,
  ↗ wave_frequency Sampled{Float64} [0.2] ForwardOrdered Irregular Points
└──────────────────────────────────────────────────────────────────────────┘
[:, :, 1]
 ↓ →          :Heave       :Pitch
  :Heave  7297.91      7297.91
  :Pitch  7297.91     10411.7

In [224]:
F_D_mh_labeled = label_output(fb, F_D_mh, omegas[1], 0.0)

┌ 2×1×1 DimArray{ComplexF64, 3} ┐
├───────────────────────────────┴──────────────────────────────────── dims ┐
  ↓ influenced_DOFs Categorical{Symbol} [:Heave, :Pitch] ForwardOrdered,
  → wave_frequency Sampled{Float64} [0.2] ForwardOrdered Irregular Points,
  ↗ wave_direction Sampled{Float64} [0.0] ForwardOrdered Irregular Points
└──────────────────────────────────────────────────────────────────────────┘
[:, :, 1]
 ↓ →              0.2
  :Heave  -284.598-4.28884im
  :Pitch  -284.598+120.541im

In [176]:
# Output

data = rand(2, 3, 5)

# 2. Define the dimensions with coordinates
da = DimArray(data, (
    X(["Heave","Surge"]),            # Longitude
    Y(1:3),            # Latitude
    Z([10, 50, 100, 250, 500]) # Vertical levels in meters
))

da[X(At("Heave")),Y(At(1)),Z(At(10))]

0.6236182552310442

In [177]:
dof_list = ["Surge","Heave","Roll"]
num_dofs = length(dof_list)
betas = range(0,pi,length=3)
num_betas = length(beta)
omegas = range(0.2,1.5,length=10)
num_omegas = length(omegas)


A = rand(num_dofs,num_dofs,num_omegas,num_betas)

A_labeled = DimArray(A, (
    Dim{:influenced_DOFs}(dof_list), 
    Dim{:radiating_DOFs}(dof_list), 
    Dim{:wave_frequencies}(omegas), 
    Dim{:wave_directions}(betas)
))


DimensionMismatch: DimensionMismatch: axes of Dim{:wave_directions} of Dim{:wave_directions}(Base.OneTo(3)) do not match array axis of Base.OneTo(1)

In [178]:
Matrix(A_labeled[:,:,1,1])

3×3 Matrix{Float64}:
 0.0       0.999821  0.837344
 0.251378  0.819991  0.660296
 0.558065  0.432412  0.173876

In [179]:
A_labeled[1,1,1,1]=0
Matrix(A_labeled[:,:,1,1])

3×3 Matrix{Float64}:
 0.0       0.999821  0.837344
 0.251378  0.819991  0.660296
 0.558065  0.432412  0.173876

In [180]:
s=[1 2; 3 4]

2×2 Matrix{Int64}:
 1  2
 3  4

In [181]:
dof_list= ["Heave", "Surge"]
function A_mat(x)
    A_data = [x^2 3*x; 1/x x]
    A_labeled = DimArray(A_data, (
        Dim{:influenced_DOFs}(dof_list), 
        Dim{:radiating_DOFs}(dof_list)
    ))
    return Matrix(A_labeled)
end

A_mat(1.5)

Zygote.jacobian(Omega + 0im)

UndefVarError: UndefVarError: `Zygote` not defined